In [1]:
# ============================================================
# 1- Database design in phpMyAdmin -> SQL codes
# ============================================================

In [2]:
# ============================================================
# 2- Phone Number Normalization
# ============================================================

import re

_PERSIAN_DIGITS = str.maketrans(
    "۰۱۲۳۴۵۶۷۸۹",
    "0123456789"
)

_ARABIC_DIGITS = str.maketrans(
    "٠١٢٣٤٥٦٧٨٩",
    "0123456789"
)


def normalize_iran_phone(value) -> str:
    """
    Return an Iranian mobile number in local 10-digit form.

    Example:
        9121234567

    Accepted formats:
        9121234567
        09121234567
        +989121234567
        00989121234567

    Persian and Arabic-Indic digits are also accepted.
    """

    if value is None:
        raise ValueError("Phone number is empty")

    # Normalize Unicode digits first.
    s = str(value).strip()
    s = s.translate(_PERSIAN_DIGITS)
    s = s.translate(_ARABIC_DIGITS)

    # Reject obvious Excel numeric artifacts.
    if re.fullmatch(r"\d+\.0+", s):
        raise ValueError(
            f"Phone number appears to have been converted to a numeric "
            f"Excel value: {value!r}"
        )

    # Keep digits and an optional leading '+'.
    s = re.sub(r"[^\d+]", "", s)

    # Convert international Iranian formats to local format.
    if s.startswith("+98"):
        s = s[3:]
    elif s.startswith("0098"):
        s = s[4:]
    elif s.startswith("98") and len(s) == 12:
        s = s[2:]
    elif s.startswith("0"):
        s = s[1:]

    # Final Iranian mobile validation.
    if not re.fullmatch(r"9\d{9}", s):
        raise ValueError(
            f"Invalid Iranian mobile number: {value!r}"
        )

    return s

In [3]:
# ============================================================
# 3- Import Excel Data into the Database
# ============================================================
from pathlib import Path

import pandas as pd
import mysql.connector

EXCEL_FILE = Path(
    "/home/saeed/repositories/contacts/import_eitaa/contacts-1-44.xlsx"
)


def get_db_connection():
    return mysql.connector.connect(
        host="127.0.0.1",
        port=3306,
        user="reza",
        password="123",
        database="eitaa_import",
        charset="utf8mb4",
    )


def import_excel():
    df = pd.read_excel(
        EXCEL_FILE,
        dtype={
            "Full Name": "string",
            "Phone Number": "string",
        },
    )

    expected = {"Full Name", "Phone Number"}

    if set(df.columns) != expected:
        raise ValueError(
            f"Expected columns {expected}, got {set(df.columns)}"
        )

    conn = get_db_connection()

    try:
        cursor = conn.cursor()

        sql = """
            INSERT INTO contacts (
                full_name,
                phone_original,
                phone_normalized,
                status
            )
            VALUES (%s, %s, %s, 'pending')
            ON DUPLICATE KEY UPDATE
                full_name = VALUES(full_name),
                phone_original = VALUES(phone_original)
        """

        processed = 0
        invalid_phones = 0
        empty_names = 0

        for _, row in df.iterrows():

            # ------------------------------------------------
            # Full name
            # ------------------------------------------------
            if pd.isna(row["Full Name"]):
                empty_names += 1
                continue

            full_name = str(row["Full Name"]).strip()

            if not full_name:
                empty_names += 1
                continue

            # ------------------------------------------------
            # Phone number
            # ------------------------------------------------
            if pd.isna(row["Phone Number"]):
                invalid_phones += 1
                print("Invalid phone number: empty Excel cell")
                continue

            phone_original = str(row["Phone Number"]).strip()

            try:
                phone_normalized = normalize_iran_phone(
                    phone_original
                )
            except ValueError as exc:
                invalid_phones += 1
                print(
                    f"Invalid phone number "
                    f"{phone_original!r}: {exc}"
                )
                continue

            # ------------------------------------------------
            # Insert/update database
            # ------------------------------------------------
            cursor.execute(
                sql,
                (
                    full_name,
                    phone_original,
                    phone_normalized,
                ),
            )

            processed += 1

        conn.commit()

        # ----------------------------------------------------
        # Verify database state after import
        # ----------------------------------------------------
        cursor.execute(
            """
            SELECT
                COUNT(*) AS total,
                SUM(status = 'pending') AS pending,
                SUM(status = 'processing') AS processing,
                SUM(status = 'success') AS success,
                SUM(status = 'skipped_existing') AS skipped_existing,
                SUM(status = 'failed') AS failed
            FROM contacts
            """
        )

        stats = cursor.fetchone()

        print()
        print("========== IMPORT RESULT ==========")
        print(f"Excel rows processed : {processed}")
        print(f"Invalid phones       : {invalid_phones}")
        print(f"Empty names          : {empty_names}")
        print()
        print(f"DB total             : {stats[0]}")
        print(f"DB pending           : {stats[1]}")
        print(f"DB processing        : {stats[2]}")
        print(f"DB success           : {stats[3]}")
        print(f"DB skipped_existing  : {stats[4]}")
        print(f"DB failed            : {stats[5]}")
        print("====================================")

    except Exception:
        conn.rollback()
        raise

    finally:
        cursor.close()
        conn.close()


if __name__ == "__main__":
    import_excel()



========== IMPORT RESULT ==========
Excel rows processed : 44
Invalid phones       : 0
Empty names          : 0

DB total             : 44
DB pending           : 44
DB processing        : 0
DB success           : 0
DB skipped_existing  : 0
DB failed            : 0


In [4]:
# ============================================================
# 4- Chrome session persistence
# ============================================================
# Clean, isolated Selenium Chrome session

from pathlib import Path
from tempfile import mkdtemp
from typing import Optional
import shutil

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.webdriver import WebDriver

# ChromeDriver

CHROMEDRIVER = (
        Path.cwd()
        / "drivers"
        / "chromedriver"
        / "chromedriver"
)

# Create a completely new temporary Chrome User Data Directory

TEMP_CHROME_PROFILE = Path(
    mkdtemp(prefix="selenium-chrome-")
)

# Close previous Selenium session, if one exists

previous_driver: Optional[WebDriver] = globals().get("driver")

if previous_driver is not None:
    try:
        previous_driver.quit()
        print("\nPrevious Selenium driver closed.")
    except Exception as e:
        print("\nPrevious Selenium driver could not be closed:")
        print(repr(e))

# Chrome options

options = Options()

options.add_argument(
    f"--user-data-dir={TEMP_CHROME_PROFILE}"
)

options.add_argument("--start-maximized")

#  Explicit manual ChromeDriver
#  Selenium Manager is not used for driver discovery.


service = Service(
    executable_path=str(CHROMEDRIVER)
)

# Create new isolated Chrome session


driver: WebDriver = webdriver.Chrome(
    service=service,
    options=options,
)

driver.set_page_load_timeout(60)


First run:

driver = create_driver()

driver.get("https://web.eitaa.com/")

Log into Eitaa manually.

Close the browser normally.

On subsequent executions Selenium will reuse that Chrome profile/session.

This is much more robust than trying to automate OTP authentication.

In [10]:
# ============================================================
# 5- Close Selenium and remove temporary Chrome profile
# ============================================================

if "driver" in globals() and driver is not None:
    try:
        driver.quit()
        print("Selenium Chrome session closed.")
    finally:
        driver = None

if "TEMP_CHROME_PROFILE" in globals():
    shutil.rmtree(
        TEMP_CHROME_PROFILE,
        ignore_errors=True
    )

    print(
        "Temporary Selenium Chrome profile removed."
    )

Selenium Chrome session closed.
Temporary Selenium Chrome profile removed.


In [5]:
# ============================================================
# 6- Selenium Selectors
# ============================================================
# To test that the selector is unique:
# allow pasting
# document.querySelectorAll(".btn-menu-item.tgico-user.rp").length

from selenium.webdriver.common.by import By


class Selectors:
    # 1. Hamburger menu
    HAMBURGER_MENU = (
        By.CSS_SELECTOR,
        ".btn-icon.btn-menu-toggle.rp.sidebar-tools-button.is-visible"
    )

    # 2. Contacts menu item
    CONTACTS_BUTTON = (
        By.XPATH,
        "//div[contains(@class, 'btn-menu-item') and .//span[normalize-space()='Contacts']]"
    )

    # 3. "+" button inside Contacts
    ADD_CONTACT_BUTTON = (
        By.CSS_SELECTOR,
        "button.tgico-add.is-visible"
    )

    # 4. First name field
    FIRST_NAME = (
        By.CSS_SELECTOR,
        'div.input-field:first-child > div.input-field-input[contenteditable="true"]'
    )

    # 5. Phone number field
    PHONE = (
        By.CSS_SELECTOR,
        'div.input-field-phone > '
        'div.input-field-input[contenteditable="true"]'
    )

    # 6. Add/Save button
    SAVE_BUTTON = (
        By.CSS_SELECTOR,
        ".popup-create-contact button.btn-primary.btn-color-primary"
    )

    # 7. Close Add Contact window ( × button )
    CLOSE_CONTACT_WINDOW = (
        By.CSS_SELECTOR,
        ".popup-create-contact button.btn-icon.popup-close"
    )

In [6]:
# ============================================================
# 7- Reliable waits
# ============================================================
#
# Use explicit waits instead of time.sleep().
#
# The purpose of this section is to provide reusable,
# reliable synchronization helpers for the Selenium code
# used in later sections.
#
# These helpers are designed for situations such as:
#   - waiting for Chrome/Eitaa elements to appear
#   - waiting for an element to become visible
#   - waiting for an element to become clickable
#   - waiting for an element/popup to disappear
#   - waiting for a specific Selenium condition
#   - handling temporary/stale DOM elements
#
# ============================================================

from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import (
    StaleElementReferenceException,
    NoSuchElementException,
    ElementClickInterceptedException,
)

# ------------------------------------------------------------
# General configuration
# ------------------------------------------------------------

WAIT_SECONDS = 30


# ------------------------------------------------------------
# 1. Create a WebDriverWait object
# ------------------------------------------------------------
#
# Keeping this in one function makes it possible to change
# the global timeout later without changing all wait helpers.

def get_wait(driver, timeout=WAIT_SECONDS):
    """
    Return a WebDriverWait instance.

    Parameters
    ----------
    driver : WebDriver
        Active Selenium WebDriver.

    timeout : int | float
        Maximum number of seconds to wait.

    Returns
    -------
    WebDriverWait
    """
    return WebDriverWait(
        driver,
        timeout,
        poll_frequency=0.2,
        ignored_exceptions=(
            NoSuchElementException,
            StaleElementReferenceException,
        ),
    )


# ------------------------------------------------------------
# 2. Wait until an element exists in the DOM
# ------------------------------------------------------------

def wait_present(driver, locator, timeout=WAIT_SECONDS):
    """
    Wait until an element is present in the DOM.

    The element does not necessarily have to be visible.

    Useful when:
        - the element exists but is not yet displayed
        - another wait will handle visibility/clickability
    """

    return get_wait(driver, timeout).until(
        EC.presence_of_element_located(locator)
    )


# ------------------------------------------------------------
# 3. Wait until an element is visible
# ------------------------------------------------------------

def wait_visible(driver, locator, timeout=WAIT_SECONDS):
    """
    Wait until an element is present and visible.

    Returns
    -------
    WebElement
    """

    return get_wait(driver, timeout).until(
        EC.visibility_of_element_located(locator)
    )


# ------------------------------------------------------------
# 4. Wait until an element is clickable
# ------------------------------------------------------------

def wait_clickable(driver, locator, timeout=WAIT_SECONDS):
    """
    Wait until an element is visible and enabled/clickable.

    This should normally be used before clicking buttons,
    links, menu items, etc.
    """

    return get_wait(driver, timeout).until(
        EC.element_to_be_clickable(locator)
    )


# ------------------------------------------------------------
# 5. Wait until an element disappears
# ------------------------------------------------------------

def wait_invisible(driver, locator, timeout=WAIT_SECONDS):
    """
    Wait until an element is no longer visible.

    This is particularly useful for:
        - closing popups
        - waiting for dialogs to disappear
        - waiting for loading overlays to disappear

    Note:
        Selenium's invisibility_of_element_located() also
        succeeds if the element is removed from the DOM.
    """

    return get_wait(driver, timeout).until(
        EC.invisibility_of_element_located(locator)
    )


# ------------------------------------------------------------
# 6. Wait until an element disappears from the DOM
# ------------------------------------------------------------

def wait_stale(driver, element, timeout=WAIT_SECONDS):
    """
    Wait until a previously obtained WebElement becomes stale.

    Useful when an existing element is replaced/removed after
    an operation such as saving a form.
    """

    return get_wait(driver, timeout).until(
        EC.staleness_of(element)
    )


# ------------------------------------------------------------
# 7. Wait for a locator to contain specific text
# ------------------------------------------------------------

def wait_text(driver, locator, text, timeout=WAIT_SECONDS):
    """
    Wait until an element contains the specified text.
    """

    return get_wait(driver, timeout).until(
        EC.text_to_be_present_in_element(
            locator,
            text,
        )
    )


# ------------------------------------------------------------
# 8. Wait for a specific attribute value
# ------------------------------------------------------------

def wait_attribute(
        driver,
        locator,
        attribute,
        value,
        timeout=WAIT_SECONDS,
):
    """
    Wait until an element has the specified attribute value.

    Example:
        wait_attribute(
            driver,
            Selectors.SOME_ELEMENT,
            "class",
            "active",
        )
    """

    return get_wait(driver, timeout).until(
        EC.attribute_to_include(
            locator,
            value,
        )
    )


# ------------------------------------------------------------
# 9. Wait until an element is selected
# ------------------------------------------------------------

def wait_selected(driver, locator, timeout=WAIT_SECONDS):
    """
    Wait until a selectable element is selected.
    """

    return get_wait(driver, timeout).until(
        EC.element_located_to_be_selected(locator)
    )


# ------------------------------------------------------------
# 10. Wait for a custom condition
# ------------------------------------------------------------

def wait_until(
        driver,
        condition,
        timeout=WAIT_SECONDS,
):
    """
    Wait until a custom Selenium condition returns a truthy
    value.

    The condition receives the driver.

    Example:

        wait_until(
            driver,
            lambda d: some_condition(d)
        )

    This is useful when Selenium's built-in expected
    conditions are not sufficient.
    """

    return get_wait(driver, timeout).until(condition)


# ------------------------------------------------------------
# 11. Safe click helper
# ------------------------------------------------------------

def wait_and_click(driver, locator, timeout=WAIT_SECONDS):
    """
    Wait until an element is clickable and then click it.

    Returns
    -------
    WebElement
        The element that was clicked.
    """

    element = wait_clickable(
        driver,
        locator,
        timeout,
    )

    element.click()

    return element


# ------------------------------------------------------------
# 12. Wait for a popup to close
# ------------------------------------------------------------

def wait_popup_closed(
        driver,
        popup_locator,
        timeout=WAIT_SECONDS,
):
    """
    Wait until a popup/dialog is no longer visible.

    This is a convenience wrapper around
    wait_invisible().

    It is intended for situations such as the Eitaa
    Add Contact popup.
    """

    return wait_invisible(
        driver,
        popup_locator,
        timeout,
    )


# ------------------------------------------------------------
# 13. Wait for an element and then enter text
# ------------------------------------------------------------

def wait_and_type(
        driver,
        locator,
        text,
        timeout=WAIT_SECONDS,
        clear_first=True,
):
    """
    Wait for an input field to become visible and type text.

    Parameters
    ----------
    driver : WebDriver
        Active Selenium driver.

    locator : tuple
        Selenium locator, for example:
            (By.CSS_SELECTOR, "...")

    text : str
        Text to enter.

    clear_first : bool
        If True, clear the input before typing.

    Returns
    -------
    WebElement
        The input element.
    """

    element = wait_visible(
        driver,
        locator,
        timeout,
    )

    if clear_first:
        element.clear()

    element.send_keys(text)

    return element


In [7]:
# ============================================================
# 8- Add Contact to Eitaa
# ============================================================

from datetime import datetime
from selenium.webdriver.common.by import By
from selenium.common.exceptions import (
    TimeoutException,
    StaleElementReferenceException,
)

# ------------------------------------------------------------
# Eitaa result message
# ------------------------------------------------------------

UNREGISTERED_MESSAGE = (
    "The person with this phone number is not registered on Eitaa yet."
)


# ------------------------------------------------------------
# Database status update helper
# ------------------------------------------------------------

def update_contact_status(
        phone_normalized,
        status,
        error_message=None,
        increment_attempt=False,
):
    """
    Update the corresponding contact in the database.

    Parameters
    ----------
    phone_normalized : str
        Normalized Iranian mobile number.

    status : str
        One of:
            pending
            processing
            success
            skipped
            failed

    error_message : str | None
        Error information for failed contacts.

    increment_attempt : bool
        If True, increase attempts by one.
    """

    conn = get_db_connection()

    try:
        cursor = conn.cursor()

        if increment_attempt:
            sql = """
                UPDATE contacts
                SET
                    status = %s,
                    attempts = attempts + 1,
                    error_message = %s,
                    completed_at = NULL
                WHERE phone_normalized = %s
            """

            cursor.execute(
                sql,
                (
                    status,
                    error_message,
                    phone_normalized,
                ),
            )

        else:
            sql = """
                UPDATE contacts
                SET
                    status = %s,
                    error_message = %s
                WHERE phone_normalized = %s
            """

            cursor.execute(
                sql,
                (
                    status,
                    error_message,
                    phone_normalized,
                ),
            )

        conn.commit()

    except Exception:
        conn.rollback()
        raise

    finally:
        cursor.close()
        conn.close()


# ------------------------------------------------------------
# Mark contact as processing
# ------------------------------------------------------------

def mark_contact_processing(phone_normalized):
    """
    Mark a contact as processing and record its start time.

    The attempt counter is increased here because every call to
    add_contact() represents one actual Eitaa attempt.
    """

    conn = get_db_connection()

    try:
        cursor = conn.cursor()

        cursor.execute(
            """
            UPDATE contacts
            SET
                status = 'processing',
                attempts = attempts + 1,
                error_message = NULL,
                started_at = NOW(),
                completed_at = NULL
            WHERE phone_normalized = %s
            """,
            (phone_normalized,),
        )

        if cursor.rowcount != 1:
            raise ValueError(
                f"Contact not found in database: {phone_normalized}"
            )

        conn.commit()

    except Exception:
        conn.rollback()
        raise

    finally:
        cursor.close()
        conn.close()


# ------------------------------------------------------------
# Mark final contact result
# ------------------------------------------------------------

def mark_contact_result(
        phone_normalized,
        status,
        error_message=None,
):
    """
    Store the final result of the current Eitaa attempt.

    Final states handled here:
        success
        skipped
        failed
    """

    conn = get_db_connection()

    try:
        cursor = conn.cursor()

        cursor.execute(
            """
            UPDATE contacts
            SET
                status = %s,
                error_message = %s,
                completed_at = NOW()
            WHERE phone_normalized = %s
            """,
            (
                status,
                error_message,
                phone_normalized,
            ),
        )

        if cursor.rowcount != 1:
            raise ValueError(
                f"Contact not found in database: {phone_normalized}"
            )

        conn.commit()

    except Exception:
        conn.rollback()
        raise

    finally:
        cursor.close()
        conn.close()


# ------------------------------------------------------------
# Detect the Eitaa "not registered" toast
# ------------------------------------------------------------

def is_unregistered_contact(driver):
    """
    Return True if Eitaa displays the exact unregistered-contact
    message.

    The toast does not have its own close button. The Add Contact
    window's close button must therefore be clicked twice.
    """

    try:
        elements = driver.find_elements(
            By.XPATH,
            (
                "//*[contains("
                "normalize-space(text()), "
                f"{UNREGISTERED_MESSAGE!r}"
                ")]"
            ),
        )

        for element in elements:
            try:
                if element.is_displayed():
                    return True
            except StaleElementReferenceException:
                continue

    except Exception:
        pass

    return False


# ------------------------------------------------------------
# Close Add Contact window
# ------------------------------------------------------------

def close_add_contact_window(driver):
    """
    Click the Add Contact window's × button.

    For an unregistered contact this function must be called
    twice:

        1. First click dismisses the toast.
        2. Second click closes the Add Contact window.
    """

    close_button = wait_clickable(
        driver,
        Selectors.CLOSE_CONTACT_WINDOW,
    )

    close_button.click()


# ------------------------------------------------------------
# Add one contact to Eitaa
# ------------------------------------------------------------

def add_contact(driver, full_name, phone_normalized):
    """
    Add exactly one contact to Eitaa.

    Parameters
    ----------
    driver : selenium.webdriver.WebDriver
        Active Selenium WebDriver.

    full_name : str
        Contact's full name.

    phone_normalized : str
        Normalized Iranian mobile number, e.g. 9121234567.

    Returns
    -------
    str
        One of:

            "success"
            "skipped"
            "failed"

    Behavior
    --------
    Registered contact:
        success -> no retry.

    Unregistered contact:
        skipped -> no retry.

    Selenium/Eitaa error:
        failed -> Section 10 decides whether another attempt
        should be made.
    """

    # --------------------------------------------------------
    # Validate basic input
    # --------------------------------------------------------

    if not full_name or not str(full_name).strip():
        raise ValueError("Contact name is empty.")

    if not phone_normalized:
        raise ValueError("Normalized phone number is empty.")

    full_name = str(full_name).strip()
    phone_normalized = str(phone_normalized).strip()

    # --------------------------------------------------------
    # Mark this contact as processing.
    #
    # attempts is incremented here.
    # Section 10 will decide whether a failed contact should
    # be attempted again.
    # --------------------------------------------------------

    mark_contact_processing(phone_normalized)

    try:

        # ====================================================
        # 1. Make sure we are on the Contacts page
        # ====================================================

        contacts_containers = driver.find_elements(
            By.CSS_SELECTOR,
            "#contacts-container",
        )

        if contacts_containers:
            contacts_container = contacts_containers[0]
        else:
            contacts_container = None

        contacts_visible = (
                contacts_container is not None
                and "active"
                in contacts_container.get_attribute("class").split()
        )

        if not contacts_visible:
            wait_and_click(
                driver,
                Selectors.HAMBURGER_MENU,
            )

            wait_and_click(
                driver,
                Selectors.CONTACTS_BUTTON,
            )

        # ====================================================
        # 2. Open Add Contact
        # ====================================================

        add_button = wait_clickable(
            driver,
            Selectors.ADD_CONTACT_BUTTON,
        )

        add_button.click()

        # ====================================================
        # 3. Fill first-name/full-name field
        # ====================================================

        wait_and_type(
            driver,
            Selectors.FIRST_NAME,
            full_name,
            clear_first=True,
        )

        # ====================================================
        # 4. Fill phone-number field
        # ====================================================

        wait_and_type(
            driver,
            Selectors.PHONE,
            phone_normalized,
            clear_first=True,
        )

        # ====================================================
        # 5. Click Save
        # ====================================================

        save_button = wait_clickable(
            driver,
            Selectors.SAVE_BUTTON,
        )

        save_button.click()

        # ====================================================
        # 6. Determine Eitaa's result
        #
        # For an unregistered number Eitaa leaves the
        # Add Contact window open and displays the toast.
        # Therefore we check for the exact toast before
        # waiting for the popup to disappear.
        # ====================================================

        try:
            get_wait(driver, WAIT_SECONDS).until(
                lambda d: (
                        is_unregistered_contact(d)
                        or not d.find_elements(
                    By.CSS_SELECTOR,
                    ".popup-create-contact.active",
                )
                )
            )

        except TimeoutException:
            raise RuntimeError(
                "Timed out while waiting for Eitaa to report "
                "the result of the Add Contact operation."
            )

        # ====================================================
        # 7. Unregistered number
        # ====================================================

        if is_unregistered_contact(driver):
            # ------------------------------------------------
            # First click:
            # dismiss the unregistered-number toast.
            # ------------------------------------------------

            close_add_contact_window(driver)

            # ------------------------------------------------
            # Second click:
            # close the Add Contact window itself.
            # ------------------------------------------------

            close_add_contact_window(driver)

            # ------------------------------------------------
            # Confirm that the Add Contact popup is closed.
            # ------------------------------------------------

            wait_popup_closed(
                driver,
                (
                    By.CSS_SELECTOR,
                    ".popup-create-contact.active",
                ),
            )

            # ------------------------------------------------
            # Store "skipped".
            #
            # This is NOT a failure and must NOT be retried.
            # ------------------------------------------------

            mark_contact_result(
                phone_normalized,
                status="skipped",
                error_message=None,
            )

            print(
                f"[SKIPPED] {full_name} - "
                f"{phone_normalized} "
                f"(not registered on Eitaa)"
            )

            return "skipped"

        # ====================================================
        # 8. Successful contact
        #
        # At this point the unregistered toast did not appear
        # and the Add Contact popup has closed.
        # ====================================================

        wait_popup_closed(
            driver,
            (
                By.CSS_SELECTOR,
                ".popup-create-contact.active",
            ),
        )

        # ----------------------------------------------------
        # Store success.
        # ----------------------------------------------------

        mark_contact_result(
            phone_normalized,
            status="success",
            error_message=None,
        )

        print(
            f"[SUCCESS] {full_name} - {phone_normalized}"
        )

        return "success"

    # ========================================================
    # 9. Any unexpected Selenium/Eitaa error
    # ========================================================

    except Exception as exc:

        error_message = (
            f"{type(exc).__name__}: {exc}"
        )

        # ----------------------------------------------------
        # Best-effort cleanup.
        #
        # If the Add Contact window is still open, try to
        # close it so Section 10 can continue with the next
        # attempt/contact.
        # ----------------------------------------------------

        try:
            popup = driver.find_elements(
                By.CSS_SELECTOR,
                ".popup-create-contact.active",
            )

            if popup:
                close_add_contact_window(driver)

                try:
                    wait_popup_closed(
                        driver,
                        (
                            By.CSS_SELECTOR,
                            ".popup-create-contact.active",
                        ),
                        timeout=5,
                    )
                except Exception:
                    pass

        except Exception:
            pass

        # ----------------------------------------------------
        # Store failed.
        #
        # The attempt was already incremented when the contact
        # was marked "processing".
        #
        # Section 10 is responsible for deciding whether this
        # failed contact should be retried.
        # ----------------------------------------------------

        try:
            mark_contact_result(
                phone_normalized,
                status="failed",
                error_message=error_message,
            )
        except Exception as db_exc:
            raise RuntimeError(
                "The Eitaa operation failed and the database "
                "could not be updated. "
                f"Eitaa error: {error_message}; "
                f"Database error: {db_exc}"
            ) from db_exc

        print(
            f"[FAILED] {full_name} - {phone_normalized}"
        )
        print(
            f"         {error_message}"
        )

        return "failed"


In [8]:
# ============================================================
# 9- Fetch Next Contact from Database
# ============================================================

def get_next_contact():
    """
    Fetch the next contact that should be processed.

    Processing order:
        1. New/pending contacts.
        2. Failed contacts with fewer than 3 attempts.

    Returns
    -------
    dict | None
        A dictionary containing:

            id
            full_name
            phone_original
            phone_normalized
            status
            attempts

        Returns None when there are no more contacts to process.

    Notes
    -----
    Section 9 only fetches a contact.

    It does NOT:
        - open Eitaa
        - call add_contact()
        - retry contacts
        - change the contact status

    Section 10 is responsible for calling add_contact() and
    handling the returned result.
    """

    conn = get_db_connection()

    try:
        cursor = conn.cursor(dictionary=True)

        cursor.execute(
            """
            SELECT
                id,
                full_name,
                phone_original,
                phone_normalized,
                status,
                attempts
            FROM contacts
            WHERE
                status = 'pending'
                OR (
                    status = 'failed'
                    AND attempts < 3
                )
            ORDER BY
                CASE
                    WHEN status = 'pending' THEN 0
                    WHEN status = 'failed' THEN 1
                END,
                id ASC
            LIMIT 1
            """
        )

        contact = cursor.fetchone()

        return contact

    finally:
        cursor.close()
        conn.close()


# ------------------------------------------------------------
# Convenience function for displaying the next contact
# ------------------------------------------------------------

def fetch_next_contact():
    """
    Fetch and display the next contact to be processed.

    This is a small wrapper around get_next_contact() and is
    useful for testing Section 9 manually before Section 10
    is implemented.

    Returns
    -------
    dict | None
        The next contact, or None if no contact remains.
    """

    contact = get_next_contact()

    if contact is None:
        print("No contacts remain to be processed.")
        return None

    print()
    print("========== NEXT CONTACT ==========")
    print(f"ID               : {contact['id']}")
    print(f"Full name        : {contact['full_name']}")
    print(f"Phone original   : {contact['phone_original']}")
    print(f"Phone normalized : {contact['phone_normalized']}")
    print(f"Status           : {contact['status']}")
    print(f"Attempts         : {contact['attempts']}")
    print("==================================")

    return contact


In [9]:
# ============================================================
# 10- Main Processing Loop
# ============================================================

# Section 10 is responsible for:
#
#   1. Fetching the next contact using Section 9.
#   2. Processing exactly one contact using Section 8.
#   3. Allowing Section 8 to record:
#        - success
#        - skipped
#        - failed
#   4. Continuing to the next contact.
#   5. Retrying failed contacts up to 3 total attempts.
#
# Section 8 performs one actual Eitaa attempt.
# Section 9 determines which contact should be processed next.
#
# No additional Selenium wait mechanism is introduced here.


MAX_ATTEMPTS = 3


def process_all_contacts(driver):
    """
    Process all eligible contacts from the database.

    Processing rules
    ----------------
    success:
        The contact was added successfully.
        No retry.

    skipped:
        The phone number is not registered on Eitaa.
        No retry.

    failed:
        The Eitaa operation failed.
        Retry while the contact has fewer than MAX_ATTEMPTS
        total attempts.

    The actual attempt counter is maintained by Section 8
    through mark_contact_processing().
    """

    processed_count = 0
    success_count = 0
    skipped_count = 0
    failed_count = 0

    print()
    print("============================================================")
    print("STARTING EITAA CONTACT IMPORT")
    print("============================================================")
    print()

    while True:

        # --------------------------------------------------------
        # 1. Fetch the next contact
        # --------------------------------------------------------
        contact = get_next_contact()

        # Section 9 returns None when no eligible contacts remain.
        if contact is None:
            break

        # --------------------------------------------------------
        # 2. Extract contact information
        # --------------------------------------------------------
        contact_id = contact["id"]
        full_name = contact["full_name"]
        phone_normalized = contact["phone_normalized"]
        current_status = contact["status"]
        previous_attempts = contact["attempts"]

        # Section 8 increments attempts when it starts the
        # actual Eitaa operation.
        current_attempt = previous_attempts + 1

        print()
        print("------------------------------------------------------------")
        print(
            f"Processing contact ID={contact_id} | "
            f"{full_name} | {phone_normalized}"
        )
        print(
            f"Status before attempt : {current_status}"
        )
        print(
            f"Attempt               : "
            f"{current_attempt}/{MAX_ATTEMPTS}"
        )
        print("------------------------------------------------------------")

        # --------------------------------------------------------
        # 3. Process exactly one contact
        # --------------------------------------------------------
        #
        # Section 8:
        #
        #   - marks the contact as processing
        #   - increments attempts
        #   - performs the Eitaa operation
        #   - stores success/skipped/failed
        #
        # Section 10 does NOT duplicate that database logic.

        try:
            result = add_contact(
                driver=driver,
                full_name=full_name,
                phone_normalized=phone_normalized,
            )

        except Exception as exc:
            # Normally add_contact() handles its own Eitaa/Selenium
            # exceptions and returns "failed".
            #
            # This catch protects the main loop from an unexpected
            # exception escaping Section 8.

            result = "failed"

            print()
            print(
                f"[MAIN LOOP ERROR] "
                f"{type(exc).__name__}: {exc}"
            )

        # --------------------------------------------------------
        # 4. Handle the result
        # --------------------------------------------------------

        if result == "success":

            success_count += 1
            processed_count += 1

            print(
                f"[COMPLETED] Contact ID={contact_id} "
                f"added successfully."
            )

            # Do not retry.
            # Continue directly to the next contact.
            continue

        elif result == "skipped":

            skipped_count += 1
            processed_count += 1

            print(
                f"[COMPLETED] Contact ID={contact_id} "
                f"skipped because the phone number is not "
                f"registered on Eitaa."
            )

            # Do not retry.
            # Continue directly to the next contact.
            continue

        elif result == "failed":

            failed_count += 1

            # ----------------------------------------------------
            # Section 8 already incremented the attempt counter.
            #
            # The current attempt number is therefore:
            #
            #     previous_attempts + 1
            # ----------------------------------------------------

            if current_attempt < MAX_ATTEMPTS:

                print(
                    f"[RETRY] Contact ID={contact_id} failed "
                    f"on attempt {current_attempt}/{MAX_ATTEMPTS}."
                )

                print(
                    f"        It will be retried after the "
                    f"remaining pending contacts are processed."
                )

                # Do not retry here directly.
                #
                # Instead, continue the loop. Section 9 will
                # determine the next eligible contact.
                continue

            else:

                processed_count += 1

                print(
                    f"[FAILED] Contact ID={contact_id} failed "
                    f"after {MAX_ATTEMPTS} attempts."
                )

                print(
                    "         No more retries will be performed."
                )

                # Continue to the next contact.
                continue

        else:
            # ----------------------------------------------------
            # Unexpected result from Section 8
            # ----------------------------------------------------
            #
            # This should never happen because Section 8 is
            # documented to return only:
            #
            #     success
            #     skipped
            #     failed
            #
            # Raise the error rather than silently continuing.

            raise ValueError(
                f"Unexpected result from add_contact(): "
                f"{result!r}"
            )

    # ============================================================
    # Final summary
    # ============================================================

    print()
    print("============================================================")
    print("EITAA CONTACT IMPORT FINISHED")
    print("============================================================")
    print(f"Successfully added : {success_count}")
    print(f"Skipped            : {skipped_count}")
    print(f"Final failed       : {failed_count}")
    print("============================================================")

In [10]:
# Start processing all eligible contacts.

process_all_contacts(driver)


STARTING EITAA CONTACT IMPORT


------------------------------------------------------------
Processing contact ID=1 | محمد کاظم آب پیکر - لیست نظام | 9188626037
Status before attempt : pending
Attempt               : 1/3
------------------------------------------------------------
[FAILED] محمد کاظم آب پیکر - لیست نظام - 9188626037
         TimeoutException: Message: 
Stacktrace:
#0 0x6196c66db87a <unknown>
#1 0x6196c600db99 <unknown>
#2 0x6196c60643dc <unknown>
#3 0x6196c6064651 <unknown>
#4 0x6196c60b0c34 <unknown>
#5 0x6196c60adeaf <unknown>
#6 0x6196c605717b <unknown>
#7 0x6196c6057f61 <unknown>
#8 0x6196c669e247 <unknown>
#9 0x6196c669ca75 <unknown>
#10 0x6196c6687b35 <unknown>
#11 0x6196c669d6ca <unknown>
#12 0x6196c6670219 <unknown>
#13 0x6196c66c5698 <unknown>
#14 0x6196c66c5835 <unknown>
#15 0x6196c66da1c3 <unknown>
#16 0x7a327429cb84 <unknown>
#17 0x7a3274329ecc <unknown>

[RETRY] Contact ID=1 failed on attempt 1/3.
        It will be retried after the remaining pending con

KeyboardInterrupt: 

In [9]:
# ============================================================
# Diagnostic Test - Locate the 30-second Timeout
# ============================================================

from selenium.webdriver.common.by import By
from selenium.common.exceptions import TimeoutException


# ------------------------------------------------------------
# Diagnostic timeout
# ------------------------------------------------------------

DIAGNOSTIC_TIMEOUT = 5


def diagnostic_step(title, function):
    """
    Execute one diagnostic step and report the exact result.
    """

    print()
    print(f"[TEST] {title}")

    try:
        result = function()

        print(f"[OK]   {title}")

        return result

    except Exception as exc:
        print(f"[FAIL] {title}")
        print(f"       Exception: {type(exc).__name__}")
        print(f"       Message  : {exc}")

        return None


# ------------------------------------------------------------
# Basic browser state
# ------------------------------------------------------------

print()
print("=" * 60)
print("EITAA SELENIUM DIAGNOSTIC TEST")
print("=" * 60)

print()
print(f"Current URL: {driver.current_url}")

print(
    f"Page title : {driver.title!r}"
)


# ------------------------------------------------------------
# Test 1 - Browser / Eitaa page
# ------------------------------------------------------------

diagnostic_step(
    "Eitaa page is available",
    lambda: wait_present(
        driver,
        (By.TAG_NAME, "body"),
        timeout=DIAGNOSTIC_TIMEOUT,
    ),
)


# ------------------------------------------------------------
# Test 2 - Contacts container
# ------------------------------------------------------------

def test_contacts_container():
    elements = driver.find_elements(
        By.CSS_SELECTOR,
        "#contacts-container",
    )

    print(
        f"       #contacts-container count: {len(elements)}"
    )

    if elements:
        for index, element in enumerate(elements):
            print(
                f"       container[{index}] "
                f"class={element.get_attribute('class')!r}"
            )

    return elements


contacts = diagnostic_step(
    "Check Contacts container",
    test_contacts_container,
)


# ------------------------------------------------------------
# Test 3 - Is Contacts currently active?
# ------------------------------------------------------------

def test_contacts_active():
    elements = driver.find_elements(
        By.CSS_SELECTOR,
        "#contacts-container",
    )

    if not elements:
        print("       Contacts container does not exist.")
        return False

    active = any(
        "active"
        in (element.get_attribute("class") or "").split()
        for element in elements
    )

    print(f"       Contacts active: {active}")

    return active


contacts_active = diagnostic_step(
    "Check whether Contacts page is active",
    test_contacts_active,
)


# ------------------------------------------------------------
# Test 4 - Hamburger menu
# ------------------------------------------------------------

def test_hamburger():
    elements = driver.find_elements(
        *Selectors.HAMBURGER_MENU
    )

    print(
        f"       Hamburger elements found: {len(elements)}"
    )

    for index, element in enumerate(elements):
        print(
            f"       hamburger[{index}] "
            f"displayed={element.is_displayed()} "
            f"enabled={element.is_enabled()}"
        )

    return elements


diagnostic_step(
    "Check Hamburger menu selector",
    test_hamburger,
)


# ------------------------------------------------------------
# Test 5 - Contacts menu button
# ------------------------------------------------------------

def test_contacts_button():
    elements = driver.find_elements(
        *Selectors.CONTACTS_BUTTON
    )

    print(
        f"       Contacts menu elements found: {len(elements)}"
    )

    for index, element in enumerate(elements):
        print(
            f"       contacts_button[{index}] "
            f"displayed={element.is_displayed()} "
            f"enabled={element.is_enabled()} "
            f"text={element.text!r}"
        )

    return elements


diagnostic_step(
    "Check Contacts menu selector",
    test_contacts_button,
)


# ------------------------------------------------------------
# Test 6 - Add Contact button
# ------------------------------------------------------------

def test_add_button():
    elements = driver.find_elements(
        *Selectors.ADD_CONTACT_BUTTON
    )

    print(
        f"       Add Contact elements found: {len(elements)}"
    )

    for index, element in enumerate(elements):
        print(
            f"       add_button[{index}] "
            f"displayed={element.is_displayed()} "
            f"enabled={element.is_enabled()} "
            f"class={element.get_attribute('class')!r}"
        )

    return elements


diagnostic_step(
    "Check Add Contact button selector",
    test_add_button,
)


# ------------------------------------------------------------
# Test 7 - Try to reach Contacts page
# ------------------------------------------------------------

def open_contacts_for_test():

    elements = driver.find_elements(
        By.CSS_SELECTOR,
        "#contacts-container",
    )

    contacts_visible = (
        bool(elements)
        and any(
            "active"
            in (element.get_attribute("class") or "").split()
            for element in elements
        )
    )

    if contacts_visible:
        print("       Contacts page is already active.")
        return True

    print("       Contacts page is not active.")
    print("       Clicking Hamburger menu...")

    wait_and_click(
        driver,
        Selectors.HAMBURGER_MENU,
        timeout=DIAGNOSTIC_TIMEOUT,
    )

    print("       Hamburger clicked.")

    print("       Clicking Contacts...")

    wait_and_click(
        driver,
        Selectors.CONTACTS_BUTTON,
        timeout=DIAGNOSTIC_TIMEOUT,
    )

    print("       Contacts clicked.")

    wait_until(
        driver,
        lambda d: any(
            "active"
            in (element.get_attribute("class") or "").split()
            for element in d.find_elements(
                By.CSS_SELECTOR,
                "#contacts-container",
            )
        ),
        timeout=DIAGNOSTIC_TIMEOUT,
    )

    print("       Contacts page became active.")

    return True


diagnostic_step(
    "Open Contacts page if necessary",
    open_contacts_for_test,
)


# ------------------------------------------------------------
# Test 8 - Add Contact button after opening Contacts
# ------------------------------------------------------------

def test_add_button_after_contacts():

    elements = driver.find_elements(
        *Selectors.ADD_CONTACT_BUTTON
    )

    print(
        f"       Add Contact buttons found: {len(elements)}"
    )

    for index, element in enumerate(elements):
        print(
            f"       add_button[{index}] "
            f"displayed={element.is_displayed()} "
            f"enabled={element.is_enabled()} "
            f"class={element.get_attribute('class')!r}"
        )

    if not elements:
        raise RuntimeError(
            "Add Contact button was not found."
        )

    return elements


diagnostic_step(
    "Check Add Contact button after Contacts is active",
    test_add_button_after_contacts,
)


# ------------------------------------------------------------
# Test 9 - Open Add Contact window
# ------------------------------------------------------------

def open_add_contact_for_test():

    wait_and_click(
        driver,
        Selectors.ADD_CONTACT_BUTTON,
        timeout=DIAGNOSTIC_TIMEOUT,
    )

    print("       Add Contact button clicked.")

    popup = wait_visible(
        driver,
        (
            By.CSS_SELECTOR,
            ".popup-create-contact.active",
        ),
        timeout=DIAGNOSTIC_TIMEOUT,
    )

    print(
        "       Add Contact popup is visible."
    )

    return popup


popup = diagnostic_step(
    "Open Add Contact window",
    open_add_contact_for_test,
)


# ------------------------------------------------------------
# Test 10 - First-name field
# ------------------------------------------------------------

diagnostic_step(
    "Check first-name field",
    lambda: wait_visible(
        driver,
        Selectors.FIRST_NAME,
        timeout=DIAGNOSTIC_TIMEOUT,
    ),
)


# ------------------------------------------------------------
# Test 11 - Phone field
# ------------------------------------------------------------

diagnostic_step(
    "Check phone-number field",
    lambda: wait_visible(
        driver,
        Selectors.PHONE,
        timeout=DIAGNOSTIC_TIMEOUT,
    ),
)


# ------------------------------------------------------------
# Test 12 - Save button
# ------------------------------------------------------------

diagnostic_step(
    "Check Save button",
    lambda: wait_clickable(
        driver,
        Selectors.SAVE_BUTTON,
        timeout=DIAGNOSTIC_TIMEOUT,
    ),
)


# ------------------------------------------------------------
# IMPORTANT:
# We deliberately DO NOT click Save.
# ------------------------------------------------------------

print()
print("=" * 60)
print("DIAGNOSTIC TEST FINISHED")
print("=" * 60)
print()
print("IMPORTANT:")
print("Save was NOT clicked.")
print("No contact was submitted.")
print()
print(
    "If the Add Contact popup is still open, close it manually."
)
print("=" * 60)




EITAA SELENIUM DIAGNOSTIC TEST

Current URL: https://web.eitaa.com/
Page title : 'Eitaa Web'

[TEST] Eitaa page is available
[OK]   Eitaa page is available

[TEST] Check Contacts container
       #contacts-container count: 0
[OK]   Check Contacts container

[TEST] Check whether Contacts page is active
       Contacts container does not exist.
[OK]   Check whether Contacts page is active

[TEST] Check Hamburger menu selector
       Hamburger elements found: 1
       hamburger[0] displayed=True enabled=True
[OK]   Check Hamburger menu selector

[TEST] Check Contacts menu selector
       Contacts menu elements found: 0
[OK]   Check Contacts menu selector

[TEST] Check Add Contact button selector
       Add Contact elements found: 0
[OK]   Check Add Contact button selector

[TEST] Open Contacts page if necessary
       Contacts page is not active.
       Clicking Hamburger menu...
       Hamburger clicked.
       Clicking Contacts...
       Contacts clicked.
       Contacts page became ac